# Colab — Experimental stacked cascade

LSTM-AE (net) → USAD (proto∥z1) → TranAD (phys∥z2) → Mahalanobis(z3)

1. Upload **`stacked_cascade_colab.zip`** to Drive `AI-TDP` (windows folder already there).  
2. Enable **GPU**.  
3. Mount Drive → **Run all**.  
4. Results: `MyDrive/AI-TDP/baselines/outputs/stacked_cascade/`

Does **not** overwrite flat `lstm_ae_*` / `usad_*` / `tranad_*` folders.

**Not** the official hierarchy — exploratory stacked AD cascade only.

Guide: `baselines/docs/COLAB_STACKED_CASCADE_GUIDE.md`

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Paths, install, unzip code (windows already on Drive)

In [ ]:
from pathlib import Path
import sys, zipfile, subprocess

WINDOWS_DIR = Path("/content/drive/MyDrive/AI-TDP/windows")
CASCADE_ZIP = Path("/content/drive/MyDrive/AI-TDP/stacked_cascade_colab.zip")
CODE_ZIP = Path("/content/drive/MyDrive/AI-TDP/baselines_code.zip")
REPO_ROOT = Path("/content/AI-TDP")
OUT_ROOT = Path("/content/drive/MyDrive/AI-TDP/baselines/outputs")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "numpy", "scikit-learn", "matplotlib",
])
REPO_ROOT.mkdir(parents=True, exist_ok=True)

need_extract = not (
    (REPO_ROOT / "baselines" / "train" / "train_stacked_cascade.py").is_file()
    and (REPO_ROOT / "baselines" / "detect" / "mahalanobis.py").is_file()
)
if need_extract:
    if CASCADE_ZIP.is_file():
        print(f"Extracting {CASCADE_ZIP} ...")
        with zipfile.ZipFile(CASCADE_ZIP, "r") as zf:
            zf.extractall(REPO_ROOT)
    elif CODE_ZIP.is_file():
        print(f"Extracting {CODE_ZIP} ...")
        with zipfile.ZipFile(CODE_ZIP, "r") as zf:
            zf.extractall(REPO_ROOT)
    else:
        raise FileNotFoundError(
            f"Upload stacked_cascade_colab.zip to {CASCADE_ZIP.parent}"
        )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert (WINDOWS_DIR / "train.npz").is_file(), f"Expected {WINDOWS_DIR}/train.npz"
assert (WINDOWS_DIR / "val.npz").is_file()
assert (WINDOWS_DIR / "test.npz").is_file()
assert (REPO_ROOT / "baselines" / "train" / "train_stacked_cascade.py").is_file()
assert (REPO_ROOT / "baselines" / "detect" / "mahalanobis.py").is_file()
assert (REPO_ROOT / "baselines" / "models" / "stacked_cascade.py").is_file()

print("Ready — using existing Drive windows")
print("WINDOWS_DIR", WINDOWS_DIR)
print("OUT_ROOT", OUT_ROOT)

## 3. Train stacked cascade

Writes to `OUT_ROOT/stacked_cascade/` only. Three stages + Mahalanobis + eval.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import runpy

OUT_DIR = OUT_ROOT / "stacked_cascade"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.argv = [
    "train_stacked_cascade.py",
    "--windows-dir", str(WINDOWS_DIR),
    "--out-dir", str(OUT_DIR),
    "--epochs", "50",
    "--patience", "8",
    "--batch-size", "32",
    "--lr", "0.001",
    "--latent-dim", "32",
    "--quantile", "0.95",
]
try:
    runpy.run_path(
        str(REPO_ROOT / "baselines" / "train" / "train_stacked_cascade.py"),
        run_name="__main__",
    )
except SystemExit as e:
    if e.code not in (0, None):
        raise
print("Training finished.")

## 4. Plot stage training curves

In [ ]:
import json
import matplotlib.pyplot as plt

OUT_DIR = OUT_ROOT / "stacked_cascade"
history = json.loads((OUT_DIR / "history.json").read_text())
config = json.loads((OUT_DIR / "config.json").read_text())

stages = [
    ("stage1_lstm", "Stage 1 LSTM-AE (net)", "MSE"),
    ("stage2_usad", "Stage 2 USAD (proto+z1)", "val score"),
    ("stage3_tranad", "Stage 3 TranAD (phys+z2)", "MSE"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, (key, title, ylab) in zip(axes, stages):
    st = history[key]
    h = st["history"]
    ax.plot(h["train_loss"], label="train")
    ax.plot(h["val_loss"], label="val")
    be = st["best_epoch"]
    if be > 0:
        ax.axvline(be - 1, color="gray", ls="--", label=f"best {be}")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.set_ylabel(ylab)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("role:", config.get("role"))
print("stage best epochs:",
      config.get("stage1_best_epoch"),
      config.get("stage2_best_epoch"),
      config.get("stage3_best_epoch"))
m = config.get("metrics") or {}
vu = m.get("val_union_test") or {}
print("roc_auc:", vu.get("roc_auc"), "f1:", vu.get("f1"))
print("out_dir:", config.get("out_dir"))

## 5. Re-run / refresh eval (optional)

Training already wrote `evaluation/stacked_cascade.json`. This refreshes the comparison table.

In [ ]:
import runpy

EVAL_DIR = OUT_ROOT / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

sys.argv = [
    "run_eval.py",
    "--model-id", "stacked_cascade",
    "--outputs-root", str(OUT_ROOT),
    "--eval-dir", str(EVAL_DIR),
    "--quantile", "0.95",
]
try:
    runpy.run_path(
        str(REPO_ROOT / "baselines" / "eval" / "run_eval.py"),
        run_name="__main__",
    )
except SystemExit as e:
    if e.code not in (0, None):
        raise

cmp = EVAL_DIR / "comparison.md"
if cmp.is_file():
    print(cmp.read_text())

## Done

Check Drive: `MyDrive/AI-TDP/baselines/outputs/stacked_cascade/best.pt`